# Mini GACS Prototype — Mood & Style Embedding Pipeline

This notebook is an interactive walkthrough of the full pipeline:

1. Download sample videos (CC0 / public domain)
2. Extract representative frames at a fixed interval
3. Compute CLIP embeddings for every frame
4. Compute pairwise cosine similarity → visualise as a heatmap
5. Run top-5 retrieval for query frames
6. Zero-shot affective scoring (text-guided CLIP probing)
7. Vibe clustering — K-means + PCA scatter plot

All visualisations are shown inline.  Run cells top-to-bottom for a
complete end-to-end demonstration.

In [ ]:
# ── Cell 1: Environment setup ─────────────────────────────────────────────
import sys, os

# Ensure the repo root is on the Python path when running from the
# notebooks/ sub-directory or JupyterHub
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

# Optional: install deps if running in a fresh cloud environment
# !pip install -q -r {repo_root}/requirements.txt

import logging
logging.basicConfig(level=logging.WARNING)  # quiet during notebook run

import numpy as np
import matplotlib
matplotlib.use('Agg')          # keep Agg so plots are still saved to disk
import matplotlib.pyplot as plt
%matplotlib inline             

print('Python', sys.version)
print('NumPy', np.__version__)
import torch; print('PyTorch', torch.__version__)

In [ ]:
# ── Cell 2: Configuration ─────────────────────────────────────────────────
from pathlib import Path

ROOT        = Path(repo_root)
VIDEOS_DIR  = ROOT / 'data' / 'videos'
FRAMES_DIR  = ROOT / 'data' / 'frames'
METADATA_DIR= ROOT / 'data' / 'metadata'
EMBED_DIR   = ROOT / 'data' / 'embeddings'
OUTPUTS_DIR = ROOT / 'outputs'

CLIP_MODEL       = 'openai/clip-vit-base-patch32'
INTERVAL_SECONDS = 1.0   # extract one frame per second
MAX_FRAMES       = 50    # cap per video
TOP_K            = 5
N_QUERIES        = 3

for d in [VIDEOS_DIR, FRAMES_DIR, METADATA_DIR, EMBED_DIR, OUTPUTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Directories ready.')

## Step 1 — Download Sample Videos

Three short CC0-licensed clips from Wikimedia Commons are fetched automatically.
If you already have videos in `data/videos/`, skip this cell.

In [ ]:
# ── Cell 3: Download ──────────────────────────────────────────────────────
import sys; sys.path.insert(0, str(ROOT))
from download_videos import download_sample_videos

downloaded = download_sample_videos(str(VIDEOS_DIR))
print(f'Downloaded / available: {len(downloaded)} video(s)')
for p in downloaded:
    print(f'  {p}')

## Step 2 — Frame Extraction

One frame is extracted every `INTERVAL_SECONDS` seconds from each video and saved as a JPEG.
Full metadata (video_id, timestamp, file_path) is written to CSV.

In [ ]:
# ── Cell 4: Extract frames ────────────────────────────────────────────────
from src.frame_extractor import process_video_directory, load_metadata

combined_csv = METADATA_DIR / 'all_frames_metadata.csv'

metadata = process_video_directory(
    video_dir=str(VIDEOS_DIR),
    frames_dir=str(FRAMES_DIR),
    metadata_dir=str(METADATA_DIR),
    interval_seconds=INTERVAL_SECONDS,
    max_frames_per_video=MAX_FRAMES,
)

print(f'Total frames extracted: {len(metadata)}')
import pandas as pd
df = pd.DataFrame(metadata)
print(df.groupby('video_id').size().rename('frames_extracted'))

In [ ]:
# ── Cell 5: Preview sample frames ─────────────────────────────────────────
from PIL import Image

sample = metadata[:6]
fig, axes = plt.subplots(1, min(6, len(sample)), figsize=(15, 3))
if len(sample) == 1:
    axes = [axes]
for ax, entry in zip(axes, sample):
    img = Image.open(entry['file_path']).convert('RGB')
    ax.imshow(img)
    ax.set_title(f"{entry['video_id']}\n@{entry['timestamp']:.1f}s", fontsize=8)
    ax.axis('off')
plt.suptitle('Sample extracted frames', fontsize=11)
plt.tight_layout()
plt.savefig(str(OUTPUTS_DIR / 'sample_frames_preview.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Saved → outputs/sample_frames_preview.png')

## Step 3 — CLIP Embeddings

We use `openai/clip-vit-base-patch32` (via HuggingFace *transformers*) to compute
L2-normalised 512-d visual embeddings for every frame.

In [ ]:
# ── Cell 6: Compute embeddings ────────────────────────────────────────────
from src.embeddings import compute_and_save_embeddings, load_embeddings
import os

npy_path = EMBED_DIR / 'embeddings.npy'

if npy_path.exists():
    print('Loading cached embeddings …')
    embeddings, index = load_embeddings(str(EMBED_DIR))
else:
    print('Computing CLIP embeddings (this may take a minute) …')
    embeddings, index = compute_and_save_embeddings(
        metadata=metadata,
        output_dir=str(EMBED_DIR),
        model_name=CLIP_MODEL,
    )

print(f'Embeddings shape : {embeddings.shape}  (dtype={embeddings.dtype})')
print(f'Index entries    : {len(index)}')

# Sanity checks
assert embeddings.ndim == 2, 'Must be 2-D'
assert not np.isnan(embeddings).any(), 'No NaNs allowed'
assert len(index) == embeddings.shape[0], 'Index/embedding length mismatch'
print('✓ All sanity checks passed.')

## Step 4 — Pairwise Cosine Similarity & Heatmap

In [ ]:
# ── Cell 7: Compute similarity matrix ────────────────────────────────────
from src.similarity import cosine_similarity_matrix, compute_inter_video_stats

sim_matrix = cosine_similarity_matrix(embeddings)
print(f'Similarity matrix shape : {sim_matrix.shape}')
print(f'Value range             : [{sim_matrix.min():.4f}, {sim_matrix.max():.4f}]')

stats = compute_inter_video_stats(sim_matrix, index)
print('\nSimilarity statistics:')
for k, v in stats.items():
    print(f'  {k}: {v:.4f}')

In [ ]:
# ── Cell 8: Heatmap ───────────────────────────────────────────────────────
from src.visualization import plot_similarity_heatmap, plot_cross_video_similarity_bar

heatmap_path = plot_similarity_heatmap(
    sim_matrix, index,
    output_path=str(OUTPUTS_DIR / 'similarity_heatmap.png'),
)
from IPython.display import Image as IPImage, display
display(IPImage(heatmap_path))

bar_path = plot_cross_video_similarity_bar(
    stats,
    output_path=str(OUTPUTS_DIR / 'cross_video_similarity_bar.png'),
)
display(IPImage(bar_path))

## Step 5 — Top-k Retrieval

For each of the `N_QUERIES` evenly-spaced query frames we retrieve the
`TOP_K` most similar frames and display them as a grid.

In [ ]:
# ── Cell 9: Top-k retrieval ───────────────────────────────────────────────
from src.similarity import batch_top_k_queries
from src.visualization import plot_top_k_grid, generate_similarity_report

n = len(index)
step = max(1, n // N_QUERIES)
query_indices = [min(i * step, n - 1) for i in range(N_QUERIES)]

query_results = batch_top_k_queries(query_indices, sim_matrix, index, top_k=TOP_K)

print('Top-k retrieval results:')
for qidx, results in query_results.items():
    meta = index[qidx]
    sims = [round(r['similarity'], 3) for r in results]
    print(f'  Query {qidx} ({meta["video_id"]} @{meta["timestamp"]}s) → {sims}')

# Display retrieval grids
for qidx, results in query_results.items():
    grid_path = plot_top_k_grid(
        qidx, results, index,
        output_path=str(OUTPUTS_DIR / f'top_k_query_{qidx}.png'),
    )
    print(f'\nQuery frame {qidx}:')
    display(IPImage(grid_path))

report_path = generate_similarity_report(
    sim_matrix, index, query_results, stats,
    output_path=str(OUTPUTS_DIR / 'similarity_report.md'),
)
print(f'\nMarkdown report: {report_path}')

## Step 6 — Zero-Shot Affective Scoring

CLIP's shared image-text space lets us probe each frame against descriptive
text prompts to score it on named affective axes (energy, warmth, complexity,
luxury, joy, tension) — **no labelled training data needed**.

The score for an axis is:

```
score_i = sim(frame_i, positive_prompt) − sim(frame_i, negative_prompt)
```

This is the core idea behind the GACS affective engine described in `REPORT.md`.

In [ ]:
# ── Cell 10: Affective scoring ────────────────────────────────────────────
from src.affective_scoring import AffectiveScorer, DEFAULT_AXES

print('Affective axes defined:')
for ax, (pos, neg) in DEFAULT_AXES.items():
    print(f'  {ax:12s}  (+) {pos}')
    print(f'{"":14s}  (−) {neg}')

scorer = AffectiveScorer(model_name=CLIP_MODEL, axes=DEFAULT_AXES)
frame_scores = scorer.score_frames(embeddings, index)

print('\nMean affective score per axis:')
for ax, scores in frame_scores.items():
    print(f'  {ax:12s}: {float(scores.mean()):+.4f}  (std={float(scores.std()):.4f})')

In [ ]:
# ── Cell 11: Affective heatmap ────────────────────────────────────────────
affect_heatmap = scorer.plot_heatmap(
    frame_scores, index,
    output_path=str(OUTPUTS_DIR / 'affective_heatmap.png'),
)
display(IPImage(affect_heatmap))

In [ ]:
# ── Cell 12: Radar chart (video-level affective profiles) ─────────────────
video_scores = scorer.score_video_level(frame_scores, index)

print('Video-level affective profiles:')
for vid, ax_scores in video_scores.items():
    vals = {k: round(v, 3) for k, v in ax_scores.items()}
    print(f'  {vid}: {vals}')

radar_path = scorer.plot_radar(
    video_scores,
    output_path=str(OUTPUTS_DIR / 'affective_radar.png'),
)
display(IPImage(radar_path))

scorer.save_scores(
    frame_scores, index,
    output_path=str(OUTPUTS_DIR / 'affective_scores.json'),
)
print('\nAll outputs saved to outputs/')

## Step 7 — Vibe Clustering

Group frames by visual style using K-means on the CLIP embeddings.
A 2-D PCA projection shows how different videos and scenes cluster
in the vibe space — the foundation of the creative library map described
in REPORT.md §2b.

```
Embeddings (N×D) ──► K-means ──► cluster labels (N,)
                  └► PCA 2-D ──► scatter plot
```

In [ ]:
# ── Cell 13: Auto-detect number of clusters ───────────────────────────────
from src.clustering import auto_n_clusters

suggested_k = auto_n_clusters(embeddings, max_k=min(10, len(index) - 1))
print(f'Suggested number of clusters: {suggested_k}')

# You can override: N_CLUSTERS = 4
N_CLUSTERS = suggested_k

In [ ]:
# ── Cell 14: Fit K-means and plot scatter ─────────────────────────────────
from src.clustering import VibeClusterer

clusterer = VibeClusterer(n_clusters=N_CLUSTERS)
labels = clusterer.fit(embeddings)

print(f'K-means fitted: {N_CLUSTERS} clusters over {len(labels)} frames')
print(f'Inertia: {clusterer._kmeans.inertia_:.4f}')

summary = clusterer.cluster_summary(labels, index)
for cid, info in summary.items():
    print(f'  Cluster {cid}: {info["size"]} frames | '
          f'videos={info["video_distribution"]}')

In [ ]:
# ── Cell 15: PCA scatter plot ─────────────────────────────────────────────
scatter_path = clusterer.plot_scatter(
    embeddings, labels, index,
    output_path=str(OUTPUTS_DIR / 'vibe_cluster_scatter.png'),
    projection='pca',
    title=f'Vibe Cluster Map — {N_CLUSTERS} clusters (PCA)',
)
display(IPImage(scatter_path))

assign_path = clusterer.save_cluster_assignments(
    labels, index,
    output_path=str(OUTPUTS_DIR / 'cluster_assignments.json'),
)
print(f'Cluster assignments: {assign_path}')

## Summary

| Output file | Description |
|-------------|-------------|
| `outputs/similarity_heatmap.png` | Full N×N pairwise cosine-similarity heatmap |
| `outputs/cross_video_similarity_bar.png` | Within vs cross-video mean similarity |
| `outputs/top_k_query_*.png` | Top-5 retrieval grids for 3 query frames |
| `outputs/similarity_report.md` | Markdown table of all retrieval results |
| `outputs/affective_heatmap.png` | Affective scores (axes × frames) heatmap |
| `outputs/affective_radar.png` | Per-video affective profile radar chart |
| `outputs/affective_scores.json` | Frame-level affective scores (JSON) |
| `outputs/vibe_cluster_scatter.png` | 2-D PCA scatter coloured by vibe cluster |
| `outputs/cluster_assignments.json` | Per-frame cluster label assignments (JSON) |

### Next Steps (from REPORT.md)

1. **Vibe–Performance Regression** — Pair vibe embeddings with historical
   CTR/ROAS data and train a ridge or MLP regression head to predict
   creative performance before launch.

2. **Retrieval-Augmented Creative Optimisation** — Store embeddings in a
   vector DB, retrieve the most similar past creatives with known performance,
   and recommend visual tweaks based on winning attributes in the neighbourhood.